In [1]:
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Normalization
from tensorflow.keras.initializers import Orthogonal
import numpy as np
import pandas as pd
import tensorflow as tf
from MockBroker import MockBroker
# import matplotlib.pyplot as plt
import plotly.graph_objects as go



models = {}
norm_layers = {}

nifty_symbols = [
    "ADANIPORTS", "ASIANPAINT", "AXISBANK", "BAJAJFINSV",
    "BAJFINANCE", "BPCL", "BRITANNIA", "CIPLA", "COALINDIA",
    "DIVISLAB", 
    "DRREDDY", "EICHERMOT", "GRASIM", "HCLTECH",
    "HDFCBANK", "HDFCLIFE", "HEROMOTOCO", "HINDALCO", 
    "HINDUNILVR",
    "ICICIBANK", "ICICIGI", "IOC", "INDUSINDBK", "INFY",
    "ITC", "JSWSTEEL", "KOTAKBANK", "LTTS", "LT",
    "MARICO",
      "MARUTI", "NESTLEIND"
]

# Load models
for symbol in nifty_symbols:
    models[symbol] = load_model(
        f'./models/{symbol}_model.keras', 
        # custom_objects={'Orthogonal': Orthogonal}
    )

data_test = {}
data_pred = {}
price = {}

threshold = 0.001
days= 500
cash =10000
loss_thres = 1000
blacklist_thres = 1

for symbol in nifty_symbols:
    # Load the test data
    price[symbol] = pd.read_csv(f'./data/{symbol}.csv').iloc[-days:, 0].values
    data_test[symbol] = pd.read_csv(f'./data/{symbol}.csv').iloc[-days:, [0,1,2,3]].values

    # Normalize the test data
    norm_l = Normalization(axis=-1)
    norm_l.adapt(data_test[symbol])  # Ensure normalization layer adapts to the test data
    data_test_n = norm_l(data_test[symbol])

    # Reshape the data for LSTM input
    data_test_n = np.reshape(data_test_n, (data_test_n.shape[0], 1, data_test_n.shape[1]))

    # Predict using the model
    data_pred[symbol] = models[symbol].predict(data_test_n)

    






16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━

In [2]:


# Dictionary to store test data and predictions


# Load the test data
# Normalization layer (reusing the one from training)
# Initialize the broker
broker = MockBroker(cash, price)



for i in range(days):
    print(f"--- Day {i} ---")
    predictions = []

    nifty_symbols = [symbol for symbol in nifty_symbols if broker.blacklistedstocks.get(symbol, 0) != 3]


    for symbol in nifty_symbols:
        predictions.append((symbol, data_pred[symbol][i][0]))

    # Sort the predictions by the prediction value
    predictions.sort(key=lambda x: x[1], reverse=True)

    # Get the top 3 symbols with the highest prediction values
    top_3_symbols = predictions[:3]
    print(top_3_symbols)
    broker.sell_off_if_thres_blacklist(i,loss_thres, blacklist_thres)


    for symbol, prediction in top_3_symbols[:1]:
        current_price = price[symbol][i]
        qty = np.floor(broker.Balance / (current_price))
        if prediction > threshold:
            broker.buy(symbol, current_price, qty)

    print(broker.holdings)


x = np.linspace(0, days, days)  # 100 points between 0 and 10
y = broker.networth

fig = go.Figure()

# Add a line plot to the figure
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='lines',
    name='Net Worth',  # Legend label
    line=dict(color='blue', width=2)  # Line styling
))

# Add title and labels to the plot
fig.update_layout(
    title='Net Worth Over Time',
    xaxis_title='Days',
    yaxis_title='Net Worth',
    template='plotly_dark'  # Optional: Add a dark theme for the plot
)

# Show the plot
fig.show()

print(broker)



--- Day 0 ---
[('COALINDIA', 0.0036495782), ('JSWSTEEL', 0.0034882904), ('LT', 0.0025744918)]
Networth: 10000
Bought 43.0 of COALINDIA at 231.64999 each
Balance: 39.050429999999324
{'COALINDIA': 43}
--- Day 1 ---
[('JSWSTEEL', 0.0040776944), ('COALINDIA', 0.0034840254), ('LT', 0.002487054)]
Networth: 9918.30043
Bought 0.0 of JSWSTEEL at 738.15002 each
Balance: 39.050429999999324
{'COALINDIA': 43, 'JSWSTEEL': 0}
--- Day 2 ---
[('COALINDIA', 0.0033901695), ('JSWSTEEL', 0.00319998), ('LT', 0.0023958427)]
Networth: 9946.25
Bought 0.0 of COALINDIA at 230.39999 each
Balance: 39.050429999999324
{'COALINDIA': 43, 'JSWSTEEL': 0}
--- Day 3 ---
[('COALINDIA', 0.00337627), ('JSWSTEEL', 0.0029081388), ('LT', 0.002395506)]
Networth: 9879.600859999999
Bought 0.0 of COALINDIA at 228.85001 each
Balance: 39.050429999999324
{'COALINDIA': 43, 'JSWSTEEL': 0}
--- Day 4 ---
[('COALINDIA', 0.0033852316), ('HCLTECH', 0.0030604778), ('JSWSTEEL', 0.0025036724)]
Sold 43 of COALINDIA at 232.39999 each
Balance: 100

Equity: 0.0
 Holdings: {'COALINDIA': 64, 'JSWSTEEL': 0, 'HCLTECH': 0, 'ICICIGI': 0, 'HDFCLIFE': 0, 'DRREDDY': 0, 'EICHERMOT': 0, 'LT': 0, 'BAJAJFINSV': 0, 'CIPLA': 0, 'ITC': 0, 'KOTAKBANK': 0, 'BRITANNIA': 0}
 Balance: 33.91057000004366
 Total: 26533.110570000044
 Highest Net Worth: 32614.65813000003
 Profit off Stocks: {'COALINDIA': -16958.2, 'JSWSTEEL': 1036.0, 'HCLTECH': 800.0, 'ICICIGI': 1292.0, 'HDFCLIFE': 3.0, 'DRREDDY': 260.0, 'EICHERMOT': 274.0, 'LT': 762.0, 'BAJAJFINSV': 688.0, 'CIPLA': 1309.0, 'ITC': 13.0, 'KOTAKBANK': 466.0, 'BRITANNIA': 95.0}


In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Normalization
from tensorflow.keras.initializers import Orthogonal
from MockBroker import MockBroker


# Initialize the broker



# Dictionary to store test data and predictions


broker = MockBroker(cash, price)


# Load the test data
# Normalization layer (reusing the one from training)




for i in range(days):
    print(f"--- Day {i} ---")
    predictions = []

    for symbol in nifty_symbols:
        predictions.append((symbol, data_pred[symbol][i][0]))

    # Sort the predictions by the prediction value
    predictions.sort(key=lambda x: x[1], reverse=True)

    # Get the top 3 symbols with the highest prediction values
    top_3_symbols = predictions[:3]
    print(top_3_symbols)
    broker.sell_off_if_thres_blacklist(i, loss_thres, blacklist_thres)



    for symbol, prediction in top_3_symbols[:3]:
        current_price = price[symbol][i]
        qty = np.floor(broker.Balance / (3*current_price))
        if prediction > threshold:
            broker.buy(symbol, current_price, qty)

    print(broker.holdings)



x = np.linspace(0, days, days)  # 100 points between 0 and 10
y = broker.networth

fig = go.Figure()

# Add a line plot to the figure
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='lines',
    name='Net Worth',  # Legend label
    line=dict(color='blue', width=2)  # Line styling
))

# Add title and labels to the plot
fig.update_layout(
    title='Net Worth Over Time',
    xaxis_title='Days',
    yaxis_title='Net Worth (USD)',
    template='plotly_dark'  # Optional: Add a dark theme for the plot
)

# Show the plot
fig.show()

print(broker)


--- Day 0 ---
[('COALINDIA', 0.0036495782), ('JSWSTEEL', 0.0034882904), ('LT', 0.0025744918)]
Networth: 10000
Bought 14.0 of COALINDIA at 231.64999 each
Balance: 6756.90014
Bought 3.0 of JSWSTEEL at 746.75 each
Balance: 4516.65014
Bought 0.0 of LT at 2093.5 each
Balance: 4516.65014
{'COALINDIA': 14, 'JSWSTEEL': 3, 'LT': 0}
--- Day 1 ---
[('JSWSTEEL', 0.0040776944), ('COALINDIA', 0.0034840254), ('LT', 0.002487054)]
Networth: 9947.6002
Bought 2.0 of JSWSTEEL at 738.15002 each
Balance: 3040.3500999999997
Bought 4.0 of COALINDIA at 229.75 each
Balance: 2121.3500999999997
Bought 0.0 of LT at 2124.0 each
Balance: 2121.3500999999997
{'COALINDIA': 18, 'JSWSTEEL': 5, 'LT': 0}
--- Day 2 ---
[('COALINDIA', 0.0033901695), ('JSWSTEEL', 0.00319998), ('LT', 0.0023958427)]
Sold 5 of JSWSTEEL at 744.70001 each
Balance: 5844.85015
Networth: 9992.04997
Bought 8.0 of COALINDIA at 230.39999 each
Balance: 4001.65023
Bought 1.0 of JSWSTEEL at 744.70001 each
Balance: 3256.95022
Bought 0.0 of LT at 2167.7 each

Equity: 0.0
 Holdings: {'COALINDIA': 12, 'JSWSTEEL': 0, 'LT': 0, 'HCLTECH': 0, 'BAJAJFINSV': 0, 'ITC': 0, 'EICHERMOT': 0, 'ADANIPORTS': 0, 'NESTLEIND': 0, 'IOC': 0, 'ICICIGI': 0, 'HDFCLIFE': 0, 'DIVISLAB': 0, 'CIPLA': 0, 'DRREDDY': 0, 'MARICO': 0, 'KOTAKBANK': 0, 'INDUSINDBK': 0, 'BRITANNIA': 0, 'ASIANPAINT': 0}
 Balance: 10262.500246000003
 Total: 15231.100246000004
 Highest Net Worth: 16589.090056
 Profit off Stocks: {'COALINDIA': -2337.6000000000004, 'JSWSTEEL': 334.0, 'LT': 192.0, 'HCLTECH': 242.0, 'BAJAJFINSV': 276.0, 'ITC': 197.0, 'EICHERMOT': 24.0, 'ADANIPORTS': 232.0, 'NESTLEIND': 15.0, 'IOC': 77.0, 'ICICIGI': 331.0, 'HDFCLIFE': 45.0, 'DIVISLAB': 0.0, 'CIPLA': 225.0, 'DRREDDY': 52.0, 'MARICO': 189.5, 'KOTAKBANK': 136.0, 'INDUSINDBK': 29.0, 'BRITANNIA': 19.0, 'ASIANPAINT': 13.0}


In [4]:

# Dictionary to store test data and predictions


broker = MockBroker(cash, price)



# Load the test data
# Normalization layer (reusing the one from training)


for i in range(days):
    print(f"--- Day {i} ---")
    predictions = []

    for symbol in nifty_symbols:
        predictions.append((symbol, data_pred[symbol][i][0]))

    # Sort the predictions by the prediction value
    predictions.sort(key=lambda x: x[1], reverse=True)

    # Get the top 3 symbols with the highest prediction values
    top_3_symbols = predictions[:3]
    print(top_3_symbols)
    broker.sell_off_if_thres_blacklist(i, loss_thres, blacklist_thres)

    
    sumprices = 0

    for symbol, prediction in top_3_symbols[:3]:
            sumprices += price[symbol][i]
    
    # print(sumprices)

    for symbol, prediction in top_3_symbols[:3]:
        current_price = price[symbol][i]
        balance_alloted = broker.Balance*current_price/sumprices
        qty = np.floor(balance_alloted / (current_price))
        if prediction > threshold:
            broker.buy(symbol, current_price, qty)

    print(broker.holdings)


x = np.linspace(0, days, days)  # 100 points between 0 and 10
y = broker.networth

fig = go.Figure()

# Add a line plot to the figure
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='lines',
    name='Net Worth',  # Legend label
    line=dict(color='blue', width=2)  # Line styling
))

# Add title and labels to the plot
fig.update_layout(
    title='Net Worth Over Time',
    xaxis_title='Days',
    yaxis_title='Net Worth (USD)',
    template='plotly_dark'  # Optional: Add a dark theme for the plot
)

# Show the plot
fig.show()

print(broker)


--- Day 0 ---
[('COALINDIA', 0.0036495782), ('JSWSTEEL', 0.0034882904), ('LT', 0.0025744918)]
Networth: 10000
Bought 3.0 of COALINDIA at 231.64999 each
Balance: 9305.05003
Bought 3.0 of JSWSTEEL at 746.75 each
Balance: 7064.80003
Bought 2.0 of LT at 2093.5 each
Balance: 2877.8000300000003
{'COALINDIA': 3, 'JSWSTEEL': 3, 'LT': 2}
--- Day 1 ---
[('JSWSTEEL', 0.0040776944), ('COALINDIA', 0.0034840254), ('LT', 0.002487054)]
Sold 2 of LT at 2124.0 each
Balance: 7125.80003
Networth: 10029.500090000001
Bought 2.0 of JSWSTEEL at 738.15002 each
Balance: 5649.49999
Bought 1.0 of COALINDIA at 229.75 each
Balance: 5419.74999
Bought 1.0 of LT at 2124.0 each
Balance: 3295.7499900000003
{'COALINDIA': 4, 'JSWSTEEL': 5, 'LT': 1}
--- Day 2 ---
[('COALINDIA', 0.0033901695), ('JSWSTEEL', 0.00319998), ('LT', 0.0023958427)]
Sold 5 of JSWSTEEL at 744.70001 each
Balance: 7019.250040000001
Sold 1 of LT at 2167.7 each
Balance: 9186.95004
Networth: 10108.55
Bought 2.0 of COALINDIA at 230.39999 each
Balance: 8726

Equity: 9178.650000000001
 Holdings: {'COALINDIA': 0, 'JSWSTEEL': 0, 'LT': 0, 'HCLTECH': 0, 'BAJAJFINSV': 0, 'ITC': 0, 'EICHERMOT': 1, 'ADANIPORTS': 0, 'NESTLEIND': 0, 'IOC': 0, 'ICICIGI': 0, 'HDFCLIFE': 0, 'DIVISLAB': 0, 'CIPLA': 3, 'DRREDDY': 0, 'MARICO': 0, 'KOTAKBANK': 0, 'INDUSINDBK': 0, 'BRITANNIA': 0, 'ASIANPAINT': 0}
 Balance: 3540.789526000047
 Total: 12719.43952600005
 Highest Net Worth: 14549.229186000033
 Profit off Stocks: {'COALINDIA': 403.0, 'JSWSTEEL': 313.0, 'LT': 642.0, 'HCLTECH': 163.0, 'BAJAJFINSV': 127.0, 'ITC': 35.0, 'EICHERMOT': -4931.0, 'ADANIPORTS': 326.0, 'NESTLEIND': 15.0, 'IOC': 10.0, 'ICICIGI': 671.0, 'HDFCLIFE': 107.0, 'DIVISLAB': 0.0, 'CIPLA': -4683.0, 'DRREDDY': 92.0, 'MARICO': 94.0, 'KOTAKBANK': 136.0, 'INDUSINDBK': 15.0, 'BRITANNIA': 19.0, 'ASIANPAINT': 13.0}
